# 💰 Farm Financial — Exploratory Data AnalysisDataset: `farm_financial_logs.csv`This notebook explores the 90-day farm financial log: data structure, missing values, profit timelines, milk-vs-profit and feed-cost-vs-profit correlations, calendar-based seasonality (day-of-week / month), and the overall profit distribution — laying the groundwork for the **Time-Series Random Forest** financial forecasting model.

In [11]:
pip install matplotlib seaborn jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.3 MB/s eta 0:00:00


# **Farm Financial EDA**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean, modern aesthetic for financial visualizations
sns.set_theme(style="darkgrid")

# Load Model 3 Dataset
df_finance = pd.read_csv('farm_financial_logs.csv')

# Explicitly convert the 'date' column to a standard datetime format
df_finance['date'] = pd.to_datetime(df_finance['date'])

print(f"💵 Financial Log Shape: {df_finance.shape} (90 Days of continuous tracking data)")
df_finance.head()

In [ ]:
print("--- Financial Data Structure ---")
print(df_finance.info())

print("\n--- Check for Missing Ledger Values ---")
print(df_finance.isnull().sum())

print("\n--- Summary Statistics of Farm Economy (PKR) ---")
df_finance.describe()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot daily volatile profits
plt.plot(df_finance['date'], df_finance['daily_profit_pkr'],
         label='Daily Net Profit', color='#9467bd', alpha=0.4, linestyle='--')

# Plot the 7-day rolling average to see the clear trajectory
plt.plot(df_finance['date'], df_finance['7_day_rolling_avg'],
         label='7-Day Rolling Trend (Smoothed)', color='#d62728', lw=2.5)

plt.title('90-Day Farm Profit Timeline Analysis (PKR) 📊', fontsize=14, pad=15)
plt.xlabel('Timeline Date', fontsize=12)
plt.ylabel('Net Return (PKR)', fontsize=12)
plt.legend(loc='upper left', frameon=True)
plt.gcf().autofmt_xdate() # Auto-rotate dates for readability
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Milk Yield vs Profit Correlation
sns.scatterplot(data=df_finance, x='total_milk_l', y='daily_profit_pkr',
                hue='sick_cow_count', palette='YlOrRd', size='sick_cow_count', ax=ax1)
ax1.set_title('Milk Volume vs. Daily Profit 🥛💰')
ax1.set_xlabel('Total Farm Milk Output (Liters)')
ax1.set_ylabel('Daily Profit (PKR)')

# 2. Feed Cost vs Profit Correlation
sns.scatterplot(data=df_finance, x='feed_cost_pkr', y='daily_profit_pkr',
                color='#1f77b4', ax=ax2)
ax2.set_title('Feed Expenditures vs. Daily Profit 🌾💰')
ax2.set_xlabel('Daily Feed & Care Costs (PKR)')
ax2.set_ylabel('Daily Profit (PKR)')

plt.tight_layout()
plt.show()

In [ ]:
# Create a multi-plot layout to check recurring calendar behaviors
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convert dates into explicit calendar text labels for grouping
df_finance['Day_of_Week'] = df_finance['date'].dt.day_name()
df_finance['Month'] = df_finance['date'].dt.month_name()

# Order days sequentially so the boxplot looks clean from Monday to Sunday
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Plot Day of Week distributions
sns.boxplot(data=df_finance, x='Day_of_Week', y='daily_profit_pkr', order=day_order, palette='Set2', ax=ax1)
ax1.set_title('Profit Distributions by Day of Week 🗓️')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=30)
ax1.set_ylabel('Net Profit (PKR)')

# Plot Month distributions
sns.boxplot(data=df_finance, x='Month', y='daily_profit_pkr', palette='Pastel1', ax=ax2)
ax2.set_title('Monthly Financial Shifts 📊')
ax2.set_ylabel('Net Profit (PKR)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

# Plot a distribution density histogram of farm cash returns
sns.histplot(data=df_finance, x='daily_profit_pkr', kde=True, color='#2ca02c', bins=20)

# Calculate critical operational markers from your financial history
mean_profit = df_finance['daily_profit_pkr'].mean()
risk_threshold = df_finance['daily_profit_pkr'].quantile(0.15) # Bottom 15% performance mark

# Draw indicator lines down your canvas grid
plt.axvline(mean_profit, color='blue', linestyle='-', lw=2, label=f'Mean Daily Profit ({mean_profit/1000:.1f}k PKR)')
plt.axvline(risk_threshold, color='red', linestyle=':', lw=2.5, label=f'Critical Risk Threshold ({risk_threshold/1000:.1f}k PKR)')

plt.title('Daily Net Profit Density & Financial Risk Profile 🩺💰', fontsize=14, pad=15)
plt.xlabel('Net Daily Profit Vector (PKR)')
plt.ylabel('Frequency Count (Days)')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()